# GENETIC RELATIONSHIP MATRIX: STEP 01

The following script filters variants required for the genetic relationship matrix. Variants were removed if they were duplicated, multi-allelic, non–single nucleotide variants, or ambiguous. The variants that passed these filters were subsequently used in the next step for pruning the microarray data from the All of Us Research Program, followed by calculation of the genetic relationship matrix.

In bcftools, a multiallelic site refers to a position in the genome where there are multiple alternative alleles observed at a single variant position. In simpler terms, a multiallelic site occurs when more than one alternative allele is present at a specific position in a set of genomic sequences.

Equivalent command: bcftools view -m 2 -M 2 multi_allelic_sites.vcf > biallelic_sites.vcf

In [ ]:
# Load packages 
from datetime import datetime
import pandas as pd 
import os

In [ ]:
# Workspace bucket 
my_bucket = os.getenv("WORKSPACE_BUCKET")
# Workspace bucket 
my_bucket

In [ ]:
# Copy PLINK .bim file from GCS to current directory
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/microarray/plink/arrays.bim .

In [ ]:
# Load PLINK .bim file into a DataFrame with specified column names
array_bim = pd.read_csv("arrays.bim", 
                     sep='\t', 
                     names=["CHROM","ID","CM","POS","ALT","REF"])

In [ ]:
arrays = [array_bim]

for array in arrays: 
    # Keep only variants with unique CHROM-POS-REF combinations (likely biallelic)
    array = array[~array[["CHROM","POS", "REF"]].duplicated(keep=False)]
    # Keep only biallelic single nucleotide variants (SNVs) where REF and ALT alleles are single bases
    array = array[(array["REF"].str.len() == 1) & (array["ALT"].str.len() == 1)]
    # Create a new column 'ALLELE_CHECK' to flag SNVs with alleles matching either CG or AT pairs (considering strand complementarity)
    array['ALLELE_CHECK'] = (
        array.apply(lambda row: sorted([row['REF'], row['ALT']]) in (['C', 'G'], ['A', 'T']), axis=1)
    )
    # Filter biallelic SNVs to keep only non-ambiguous variants (exclude C/G and A/T pairs)
    array = array[array["ALLELE_CHECK"] == False]
    
    # Display counts of REF-ALT pairs among non-ambiguous biallelic SNVs
    print("=========================================================================")
    print("=========================================================================")
    print("Check the presence of non-ambiguous biallelic single nucleotide variants:")
    print(array[["REF", "ALT"]].value_counts())
    print("=========================================================================")
    print("=========================================================================")
    
    # Check for duplicated positions 
    if not array[array[["CHROM","POS"]].duplicated()].empty:
        print("Presence of duplicated position ....")
        break 
    
    # Check for duplicated reference alleles
    if not array[array[["CHROM","POS", "REF"]].duplicated()].empty:
        print("Presence of duplicated reference allele ....")
        break 
        
    # Check for duplicated multiallelic variants
    if not array[array[["CHROM","POS", "REF", "ALT"]].duplicated()].empty:
        print("Presence of duplicated multiallelic variant ....")
        break 
        
    # Check for duplicated variants
    if not array[array["ID"].duplicated()].empty:
        print("Presence of duplicated variant ....")
        break 
    
    # Check for single nucleotide variants in reference allele  
    if not (array["REF"].str.len() == 1).all():
        print("Presence of non-single nucleotide variants in reference allele ....")
        break 
    
    # Check for single nucleotide variants in alternative allele  
    if not (array["ALT"].str.len() == 1).all():
        print("Presence of non-single nucleotide variants in alternative allele ....")
        break 
     
    # Check for amgious biallelic variants 
    if (array["ALLELE_CHECK"] == True).any():
        print("Presence of amgious biallelic variant ....")
        break 
         
    # Save the list of non-ambiguous biallelic SNV IDs to 
    # a tab-separated text file without header or index
    array["ID"].to_csv("non_ambiguous_biallelic_snvs.txt",
                       sep ='\t', 
                       header=None,
                       index=None)

## SAVE RESULTS

In [ ]:
# Copy the local file 'non_ambiguous_biallelic_snvs.txt' to a Google Cloud Storage bucket path
!gsutil cp non_ambiguous_biallelic_snvs.txt {my_bucket}/data/saige_step00/pruning_files/

In [ ]:
# List files in the specified Google Cloud Storage bucket directory
!gsutil ls {my_bucket}/data/saige_step00/pruning_files/

In [ ]:
# Count the number of lines in the uploaded file on Google Cloud Storage
!gsutil cat {my_bucket}/data/saige_step00/pruning_files/non_ambiguous_biallelic_snvs.txt | wc -l

## DOWNLOAD HIGH LD REGION FILES

In [ ]:
# Clone the 'plinkQC' repository from CRAN's GitHub mirror to your local environment
!git clone https://github.com/cran/plinkQC.git

In [ ]:
# Upload high linkage disequilibrium (LD) region files (txt and bed) from the plinkQC package to your GCS bucket
!gsutil cp plinkQC/inst/extdata/high-LD-regions-hg38-GRCh38.txt {my_bucket}/data/saige_step00/pruning_files/
!gsutil cp plinkQC/inst/extdata/high-LD-regions-hg38-GRCh38.bed {my_bucket}/data/saige_step00/pruning_files/

In [ ]:
# List the contents of the pruning_files directory in your Google Cloud Storage bucket
!gsutil ls {my_bucket}/data/saige_step00/pruning_files/